In [3]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error

# Load dataset
df = pd.read_csv("../car_fuel_efficiency_2026.csv")

# Keep only required columns
df = df[
    [
        "engine_displacement",
        "horsepower",
        "vehicle_weight",
        "model_year",
        "fuel_efficiency_mpg",
    ]
]

# --------------------------------------------------
# Q1: Missing values
# --------------------------------------------------

print("\nMissing values:")
print(df.isna().sum())

# --------------------------------------------------
# Q2: Median horsepower
# --------------------------------------------------

print("\nMedian horsepower:")
print(df["horsepower"].median())

# --------------------------------------------------
# Split function
# --------------------------------------------------

def split_data(df, seed):
    n = len(df)

    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].copy()
    df_val = df.iloc[idx[n_train:n_train + n_val]].copy()
    df_test = df.iloc[idx[n_train + n_val:]].copy()

    return df_train, df_val, df_test


# --------------------------------------------------
# Helper function for linear regression
# --------------------------------------------------

features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year",
]

target = "fuel_efficiency_mpg"


def train_lr(df_train, df_val, fill_value):
    train = df_train.copy()
    val = df_val.copy()

    train[features] = train[features].fillna(fill_value)
    val[features] = val[features].fillna(fill_value)

    X_train = train[features]
    y_train = train[target]

    X_val = val[features]
    y_val = val[target]

    model = LinearRegression()
    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    return root_mean_squared_error(y_val, pred)


# --------------------------------------------------
# Q3: Fill missing values with 0 vs mean
# --------------------------------------------------

df_train, df_val, df_test = split_data(df, 42)

# Fill with 0
rmse_zero = train_lr(df_train, df_val, 0)

# Mean must be calculated using TRAINING data only
train_mean = df_train["engine_displacement"].mean()
rmse_mean = train_lr(df_train, df_val, train_mean)

print("\nQ3:")
print("RMSE with 0:", round(rmse_zero, 3))
print("RMSE with mean:", round(rmse_mean, 3))


# --------------------------------------------------
# Q4: Regularization
# --------------------------------------------------

print("\nQ4:")

train = df_train.copy()
val = df_val.copy()

train[features] = train[features].fillna(0)
val[features] = val[features].fillna(0)

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:

    model = Ridge(alpha=r)
    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    rmse = root_mean_squared_error(y_val, pred)

    print(r, round(rmse, 4))


# --------------------------------------------------
# Q5: Different random seeds
# --------------------------------------------------

print("\nQ5:")

scores = []

for seed in range(10):

    train, val, test = split_data(df, seed)

    train[features] = train[features].fillna(0)
    val[features] = val[features].fillna(0)

    model = LinearRegression()

    model.fit(
        train[features],
        train[target]
    )

    pred = model.predict(val[features])

    rmse = root_mean_squared_error(
        val[target],
        pred
    )

    scores.append(rmse)

    print(seed, rmse)

std = np.std(scores)

print("Scores:", scores)
print("Standard deviation:", std)
print("Rounded:", round(std, 3))


# --------------------------------------------------
# Q6: Seed 9, train + validation, Ridge r=0.001
# --------------------------------------------------

print("\nQ6:")

train, val, test = split_data(df, 9)

# Combine train and validation
df_train_full = pd.concat(
    [train, val],
    ignore_index=True
)

df_train_full[features] = df_train_full[features].fillna(0)
test[features] = test[features].fillna(0)

model = Ridge(alpha=0.001)

model.fit(
    df_train_full[features],
    df_train_full[target]
)

pred = model.predict(test[features])

rmse_test = root_mean_squared_error(
    test[target],
    pred
)

print("Test RMSE:", rmse_test)
print("Rounded:", round(rmse_test, 3))


Missing values:
engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

Median horsepower:
254.0

Q3:
RMSE with 0: 2.205
RMSE with mean: 2.206

Q4:
0 2.2053
0.01 2.2053
0.1 2.2053
1 2.2053
5 2.2053
10 2.2053
100 2.2053

Q5:
0 2.2392041430507534
1 2.202112821083734
2 2.163419778753079
3 2.204358876852931
4 2.2018800943317713
5 2.245785150908142
6 2.2697212079516524
7 2.190794599935421
8 2.2166112016944193
9 2.20703659282341
Scores: [2.2392041430507534, 2.202112821083734, 2.163419778753079, 2.204358876852931, 2.2018800943317713, 2.245785150908142, 2.2697212079516524, 2.190794599935421, 2.2166112016944193, 2.20703659282341]
Standard deviation: 0.028781274078199606
Rounded: 0.029

Q6:
Test RMSE: 2.235774702813794
Rounded: 2.236
